### imports and pandas settings

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [ ]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [ ]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [ ]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "SMOTE" / "base_predictors" / "XGBoost_thres0.5_2026-05-07" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [ ]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [ ]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [ ]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [ ]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.98,,,,,
1,0,cf_1,,,,,,,18.3517,,,0.9062,1,False,0.0274,0.06
2,0,cf_2,,,,7,,,18.9745,,,0.8756,2,False,0.0274,0.3401
3,0,cf_3,2,,,,,,18.9619,,,0.8762,2,False,0.0274,0.0563
4,0,cf_4,,,,7,,,18.884,,,0.88,2,False,0.0274,0.3401
5,0,cf_5,,,6,,,,18.1407,,,0.9166,2,False,0.0274,0.0842
6,0,cf_6,3,,,,,,17.1903,,,0.9633,2,False,0.0274,0.0192
7,0,cf_7,,,,,,,16.4429,5,,1.0,2,True,0.0274,0.0021
8,0,cf_8,,,,7,,,18.9745,7,,0.8756,3,False,0.0274,0.0698
9,0,cf_9,1,,,5,,,18.9619,,,0.8762,3,False,0.0274,0.0183


### Filtering Valid CFs

In [ ]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.98,,,,,
9,0,cf_7,,,,,,,16.4429,5,,1.0,2,True,0.0274,0.0021
1,1,xin,3,4,1,2,3,0,22.3757,0,1.01,,,,,
10,1,cf_1,,,4,,,,22.3757,4,,0.8796,3,True,0.6924,0.3264
11,1,cf_3,,,,,1,,22.3757,1,,0.8796,3,True,0.6924,0.2375
12,1,cf_4,,,,3,1,,22.3757,,,0.8796,3,True,0.6924,0.2812
13,1,cf_5,2,,,,1,,22.3757,,,0.8796,3,True,0.6924,0.1514
14,1,cf_6,,,,3,2,,22.3757,2,,0.8796,4,True,0.6924,0.1566
15,1,cf_7,,2,,4,,,22.3757,5,,0.8796,4,True,0.6924,0.2075
16,1,cf_8,2,,,,1,,22.3757,7,,0.8796,4,True,0.6924,0.2118


### select one CF

In [ ]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.98,,,,,
9,0,cf_7,,,,,,,16.4429,5,,1.0,2,True,0.0274,0.0021
1,1,xin,3,4,1,2,3,0,22.3757,0,1.01,,,,,
10,1,cf_1,,,4,,,,22.3757,4,,0.8796,3,True,0.6924,0.3264
2,2,xin,5,3,1,1,4,0,22.694,7,1.08,,,,,
11,2,cf_8,1,,,3,1,,22.68,,,0.8753,4,True,0.0454,0.0146
3,3,xin,3,3,6,6,2,0,24.3418,1,1.09,,,,,
12,3,cf_8,,,,,,,23.0,3,,0.9071,2,True,0.4508,0.2113
4,4,xin,5,4,2,7,1,0,21.2585,3,1.24,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.9868,1,False,0.0149,0.0149


In [ ]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_7 ---
Predicted risk: 0.0021
Valid: True
Changes:
  bmi: 18.9866 → 16.4429
  dosprt: 0 → 5


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.3264
Valid: True
Changes:
  cgtsmok: 1 → 4
  dosprt: 0 → 4

--- cf_3 ---
Predicted risk: 0.2375
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_4 ---
Predicted risk: 0.2812
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_5 ---
Predicted risk: 0.1514
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_6 ---
Predicted risk: 0.1566
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 2
  dosprt